# Projects: predicting the properties of inorganic nanomaterials

Welcome to the project of the Deep Learning course!

First of all, the projects suggested here are only that, suggestions. They are for those of you who do not have an idea for a deep learning project. If you do, you are welcome to use your own data and train your own model.

Now, the project suggestions. We propose a project to do **regression** and another to do **classification**. We are going to use the [**MatBench**](https://github.com/materialsproject/matbench) benchmark datasets (Dunn, A., Wang, Q., Ganose, A., Dopp, D., & Jain, A. (2020). Benchmarking materials property prediction methods: the Matbench test set and Automatminer reference algorithm. npj Computational Materials, 6(1), 138.). We will extract them from the `matminer` library, and they are these two:

- **Matbench MP gap (Regression)**: Matbench v0.1 test dataset for predicting DFT formation energy from structure.
- **Matbench MP is metal (Classification)**: Matbench v0.1 test dataset for predicting DFT metallicity from structure.

In [ ]:
# Only run this if you are on Google Colab. Otherwise, run "uv sync" on terminal to install it on your local environment
!pip install matminer

## Featurization of the data

In both cases, the dataset only has the chemical formula string (e.g. "ZnS", "GaAs", etc.), and the target value associated to it. Neural networks need numbers, not strings.

We are going to use **MAGPIE (Materials Agnostic Platform for Informatics and Exploration** features: for each element in the compound, we loop up 22 atomic properties (atomic number, electronegativity, atomic radius, etc.) and compute 6 statistics across all elements weighted by stoichiometry. This yields **132 numerical features** per composition.

This approach captures physically meaningful information.

The following code is shared for both projects. It performs the actions described in the previous paragraphs, and allows you to build the appropriate dataloaders with the

In [9]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.conversions import StrToComposition

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ── Shared featurization ────────────────────────────────────────────────────

def featurize_compositions(df: pd.DataFrame,
                           composition_col: str = "composition") -> pd.DataFrame:
    """
    Convert a column of composition strings (e.g. 'ZnS') into
    132 numerical MAGPIE features.

    Returns a new DataFrame with feature columns only (no strings).
    """
    # Step 1: convert string → pymatgen Composition object
    stc = StrToComposition(target_col_id="composition_obj")
    df = stc.featurize_dataframe(df, composition_col, ignore_errors=True)

    # Step 2: compute MAGPIE features from Composition objects
    ep = ElementProperty.from_preset("magpie")
    df = ep.featurize_dataframe(df, col_id="composition_obj", ignore_errors=True)

    # Keep only the numeric feature columns
    feature_cols = ep.feature_labels()
    df = df.dropna(subset=feature_cols)   # drop any failed entries

    return df, feature_cols

# ── Shared PyTorch Dataset ──────────────────────────────────────────────────

class MaterialsDataset(Dataset):
    """
    Simple tabular dataset for materials property prediction.

    X: float32 tensor of shape (n_samples, n_features)
    y: float32 tensor of shape (n_samples, 1)
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loader(X, y, batch_size=64, shuffle=True, drop_last=False):
    """Build train and validation DataLoaders."""
    dataset = MaterialsDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)
    return loader

# Regression Project: predicting experimental band gap

**Task**: given only the chemical formula of an inorganic compound, predict its experimentally measured band gap in eV.

**Dataset**: `matbench_expt_gap`

## Definition of the Dataset and DataLoaders

In [10]:
# Load dataset

df_gap = load_dataset("matbench_expt_gap")
print(df_gap.head())
print(f"\nShape: {df_gap.shape}")
print(f"Target column - 'gap expt': \n{df_gap['gap expt'].describe()}")

         composition  gap expt
0           Ag(AuS)2      0.00
1         Ag(W3Br7)2      0.00
2   Ag0.5Ge1Pb1.75S4      1.83
3  Ag0.5Ge1Pb1.75Se4      1.51
4             Ag2BBr      0.00

Shape: (4604, 2)
Target column - 'gap expt': 
count    4604.000000
mean        0.975951
std         1.445034
min         0.000000
25%         0.000000
50%         0.000000
75%         1.812500
max        11.700000
Name: gap expt, dtype: float64


In [11]:
# Featurization with the defined functions at the beginning

df_gap, feature_cols = featurize_compositions(df_gap)
X = df_gap[feature_cols].values.astype(np.float32)
y = df_gap["gap expt"].values.astype(np.float32)

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

StrToComposition:   0%|          | 0/4604 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/4604 [00:00<?, ?it/s]

Feature matrix shape: (4604, 132)
Target vector shape: (4604,)


In [14]:
# Definition of data split and loaders

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True, drop_last=True)
val_loader   = make_loader(X_val, y_val, batch_size=64, shuffle=False, drop_last=False)
test_loader  = make_loader(X_test, y_test, batch_size=64, shuffle=False, drop_last=False)

In [15]:
print(f"Train samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")

Train samples: 2946
Validation samples: 737
Test samples: 921


## Training

The data is set up for you. From here, you will have to build a simple model to do the regression, decide on a loss function and an optimizer. Advice:
- For the model: remember you have an input of 132 features and an output of a single value. You can have more than a single hidden layer. Remember that between layers goes the activation function.
- For the loss function: it is a regression. Think what loss function you can use for that case.

In [16]:
# Define the model

import torch.nn as nn

class BandGapNet(nn.Module):
  """
  Feed-forward network for band gap regression
  """
  def __init__(self, n_features: int):
    super(BandGapNet, self).__init__()
    # Code here

  def forward(self, x):
    # Code here

In [24]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = ...
# optimizer = ...
# criterion = ...

In [18]:
def train_epoch(model, loader, optimizer, criterion, device):
  model.train()
  total_loss = 0.0
  for batch in loader:
    # Complete the loop to process each training batch
    # ...
  return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
  model.eval()
  total_loss = 0.0
  with torch.no_grad():
    for batch in loader:
      # Complete the loop to process each validation batch
      # ...
  return total_loss / len(loader)

In [ ]:
history = {"train_loss": [], "val_loss": []}

N_EPOCHS = 80
for epoch in range(N_EPOCHS):
  # Remember, for each epoch you have to perform a train_epoch and an evaluation epoch
  # train_loss = ...
  # val_loss = ...
  # history["train_loss"] =
  # history["val_loss"] =

# Plot graphs once finished with matplotlib to see the evolution of the losses

# Classification project: metal or non-metal?

**Task**: given only the chemical formula, predict whether a compound is a metal
(band gap = 0) or a semiconductor/insulator (band gap > 0).

**Dataset**: `matbench_expt_is_metal`

## Definition of Dataset and DataLoaders

In [25]:
df_metal = load_dataset("matbench_expt_is_metal")
print(df_metal.head())
print(f"\nShape: {df_metal.shape}")
print(f"\nClass balance:\n{df_metal['is_metal'].value_counts()}")

Fetching matbench_expt_is_metal.json.gz from https://ml.materialsproject.org/projects/matbench_expt_is_metal.json.gz to /usr/local/lib/python3.12/dist-packages/matminer/datasets/matbench_expt_is_metal.json.gz


Fetching https://ml.materialsproject.org/projects/matbench_expt_is_metal.json.gz in MB: 0.034816MB [00:00,  3.42MB/s]                 

         composition  is_metal
0           Ag(AuS)2      True
1         Ag(W3Br7)2      True
2   Ag0.5Ge1Pb1.75S4     False
3  Ag0.5Ge1Pb1.75Se4     False
4             Ag2BBr      True

Shape: (4921, 2)

Class balance:
is_metal
False    2470
True     2451
Name: count, dtype: int64


In [26]:
df_metal, feature_cols = featurize_compositions(df_metal, composition_col="composition")

X_cls = df_metal[feature_cols].values.astype(np.float32)
y_cls = df_metal["is_metal"].astype(int).values.astype(np.float32)

print(f"Feature matrix shape: {X_cls.shape}")
print(f"Class balance — metals: {y_cls.mean():.1%} | non-metals: {1-y_cls.mean():.1%}")

StrToComposition:   0%|          | 0/4921 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/4921 [00:00<?, ?it/s]

Feature matrix shape: (4921, 132)
Class balance — metals: 49.8% | non-metals: 50.2%


In [27]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42,
    stratify=y_cls    # preserve class balance in both splits
)

X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(
    X_train_c, y_train_c, test_size=0.2, random_state=42,
    stratify=y_train_c
)

scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(X_train_c)
X_val_c   = scaler_c.transform(X_val_c)
X_test_c  = scaler_c.transform(X_test_c)

train_loader_c = make_loader(X_train_c, y_train_c, batch_size=64, shuffle=True, drop_last=True)
val_loader_c   = make_loader(X_val_c, y_val_c, batch_size=64, shuffle=False, drop_last=False)
test_loader_c  = make_loader(X_test_c, y_test_c, batch_size=64, shuffle=False, drop_last=False)

In [28]:
print(f"Train samples: {len(train_loader_c.dataset)}")
print(f"Validation samples: {len(val_loader_c.dataset)}")
print(f"Test samples: {len(test_loader_c.dataset)}")

Train samples: 3148
Validation samples: 788
Test samples: 985


## Training

The data is set up for you. From here, you will have to build a simple model to do classification, decide on a loss function and an optimizer. Advice:
- For the model: remember you have an input of 132 features and an output of a single value (in this case, the value is always between 0 and 1, since it is the probability of the input of being a metal). You can have more than a single hidden layer. Remember that between layers goes the activation function.
- For the loss function: it is a classification. That means, as stated before, that the model output has to be constrained to the range (0, 1). Look up what the [`Softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html) operation is, and see how you can include it either in the model or the loss function.

In [29]:
class MetalClassifierNet(nn.Module):
    """
    Feed-forward network for binary classification.
    """

    def __init__(self, n_features: int, dropout: float = 0.3):
        super().__init__()
        # Code here

    def forward(self, x):
        # Code here



In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model_cls = ...
# optimizer_cls = ...
# criterion_cls = ...

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
  model.train()
  total_loss = 0.0
  for batch in loader:
    # Complete the loop to process each training batch
    # ...
  return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
  model.eval()
  total_loss = 0.0
  with torch.no_grad():
    for batch in loader:
      # Complete the loop to process each validation batch
      # ...
  return total_loss / len(loader)

In [ ]:
history = {"train_loss": [], "val_loss": []}

N_EPOCHS = 80
for epoch in range(N_EPOCHS):
  # Remember, for each epoch you have to perform a train_epoch and an evaluation epoch
  # train_loss = ...
  # val_loss = ...
  # history["train_loss"] =
  # history["val_loss"] =

# Plot graphs once finished with matplotlib to see the evolution of the losses